# Three decode paths: memory and speed

| script | path | needs |
|---|---|---|
| `run_vllm.py` | vLLM server over HTTP | server **UP** |
| `run_generate.py` | `model.generate(output_logits=True)` | server **DOWN** |
| `run_manual.py` | hand-written decode loop | server **DOWN** |

Each writes `results/<path>.json`. They are separate processes because the server
reserves VRAM at startup, so it cannot be up while the in-process paths run.

```
# with a vLLM server up (e.g. dxo-orchestration: docker compose --profile gpu up -d)
python run_vllm.py

# with every vLLM server stopped
python run_generate.py
python run_manual.py
```

**Read the memory column carefully.** `decode_mib` = peak allocator high-water mark
minus the level before decoding (i.e. the weights). That isolates what decoding cost
— but only for the two in-process paths. vLLM's allocator is in another process and
claims its slab at startup, so its number is an nvidia-smi sample, not a comparable
peak. Speed is comparable across all three: same model, same prompt, greedy decoding.

In [ ]:
import json
from pathlib import Path

RESULTS = Path("results")  # sits next to this notebook
ORDER = ["vllm", "generate", "manual"]

data = {p.stem: json.loads(p.read_text()) for p in RESULTS.glob("*.json")}
found = [k for k in ORDER if k in data]
missing = [k for k in ORDER if k not in data]

for key in found:
    meta = data[key]["meta"]
    print(f"{key:<9} {data[key]['label']:<34} model={data[key]['model']}")
    print(f"{'':<9} saved {meta['saved_at']}  decoding={meta.get('decoding')}")
    if meta.get("other_gpu_mib_at_start"):
        print(f"{'':<9} NOTE other processes held {meta['other_gpu_mib_at_start']:.0f} MiB")
if missing:
    print(f"\nnot run yet: {missing}  (the cells below just skip them)")

## Speed

All three generate the same tokens (greedy), so tokens/sec is directly comparable.

In [ ]:
budgets = sorted({r["budget"] for key in found for r in data[key]["runs"]})


def cell(key, budget, field):
    for run in data[key]["runs"]:
        if run["budget"] == budget:
            return run.get(field)
    return None


def table(field, unit, fmt="{:.1f}"):
    head = f"{'budget':>7} " + " ".join(f"{k:>14}" for k in found)
    print(head)
    print("-" * len(head))
    for budget in budgets:
        row = f"{budget:>7} "
        for key in found:
            value = cell(key, budget, field)
            row += f"{fmt.format(value) if value is not None else '-':>14} "
        print(row)
    print(f"({unit})")


table("tok_per_s", "tokens per second, higher is better")
print()
table("latency_s", "wall seconds for the whole request", "{:.2f}")

## Memory

`decode_mib` is what decoding cost on top of the weights. The question is how it
scales with the number of steps.

In [ ]:
table("decode_mib", "MiB above the weights; vLLM's is cross-process, see note below")

if "generate" in found and "manual" in found:
    print("\ngenerate() minus manual loop, and what one logit row costs:")
    vocab = data["manual"]["meta"]["vocab"]
    print(f"    {'budget':>7} {'extra MiB':>11} {'per step':>10} {'fp32 row':>10}")
    for budget in budgets:
        g, m = cell("generate", budget, "decode_mib"), cell("manual", budget, "decode_mib")
        if g is None or m is None:
            continue
        steps = cell("manual", budget, "tokens") or 1
        print(f"    {budget:>7} {g - m:>11.1f} {(g - m) / steps:>10.3f} {vocab * 4 / 2**20:>10.3f}")
    print("\n'per step' landing on 'fp32 row' is the mechanism: generate() keeps one")
    print("full-vocab float32 logit row per step until the loop ends.")

In [ ]:
# Both in-process paths also pay for the KV cache, which grows with sequence length
# no matter how the loop is written. Separating the two terms is what makes the
# comparison honest: only the logits term differs between the paths.
if "manual" in found:
    meta = data["manual"]["meta"]
    n_prompt = meta["prompt_tokens"]
    print(f"{'budget':>7} {'seq len':>8} {'manual decode':>14} {'of which KV*':>13}")
    for budget in budgets:
        m = cell("manual", budget, "decode_mib")
        if m is None:
            continue
        seq = n_prompt + (cell("manual", budget, "tokens") or 0)
        # Qwen3-4B: 36 layers, 8 KV heads, head_dim 128, bf16 -> 2 bytes.
        kv = 36 * 2 * 8 * 128 * seq * 2 / 2**20
        print(f"{budget:>7} {seq:>8} {m:>14.1f} {kv:>13.1f}")
    print("\n*computed from the architecture, not measured. The remainder is transient")
    print(" activations plus allocator slack.")

In [ ]:
import matplotlib.pyplot as plt

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
for key in found:
    runs = [r for r in data[key]["runs"] if r.get("tok_per_s")]
    left.plot([r["budget"] for r in runs], [r["tok_per_s"] for r in runs], "o-", label=key)
    mem = [r for r in data[key]["runs"] if r.get("decode_mib") is not None]
    if key != "vllm":  # cross-process number, not on the same axis
        right.plot([r["budget"] for r in mem], [r["decode_mib"] for r in mem], "o-", label=key)

left.set(xlabel="max_tokens", ylabel="tokens / s", title="Speed")
left.set_ylim(bottom=0)
right.set(xlabel="max_tokens", ylabel="MiB above weights", title="Decode memory (in-process only)")
right.set_ylim(bottom=0)
for axis in (left, right):
    axis.grid(alpha=0.3)
    axis.legend()
plt.tight_layout()
plt.show()

## Why vLLM's memory number is not on the same axis

vLLM claims `--gpu-memory-utilization` of the card when it starts and works inside
that slab, mostly as KV cache. So its per-request delta is ~0 — which is not vLLM
winning the memory comparison, it is the cost being paid up front and becoming
invisible per request. `run_vllm.py` samples the server process with nvidia-smi and
records the caveat in its own `meta`.

The comparison vLLM genuinely contributes here is speed.

In [ ]:
if "vllm" in found:
    meta = data["vllm"]["meta"]
    print(f"server pid {meta.get('server_pid')}  fingerprint {meta.get('fingerprint')}")
    print(f"caveat: {meta.get('memory_caveat')}")
    for run in data["vllm"]["runs"]:
        print(
            f"  budget={run['budget']:>4} server held {run['baseline_mib']} -> "
            f"{run['peak_mib']} MiB  (delta {run['decode_mib']})"
        )
else:
    print("run_vllm.py has not been run; start the server and run it to fill this in.")